In [1]:
import polars as pl
from pathlib import Path
from datetime import datetime, timezone

DATA_DIR = Path('../data/processed')
WIDE = DATA_DIR / 'wide'

trainA = pl.read_parquet(WIDE / 'trainA.parquet')
valA   = pl.read_parquet(WIDE / 'valA.parquet')
testA  = pl.read_parquet(WIDE / 'testA.parquet')
trainB = pl.read_parquet(WIDE / 'trainB.parquet')
valB   = pl.read_parquet(WIDE / 'valB.parquet')
testB  = pl.read_parquet(WIDE / 'testB.parquet')

print(f'trainA: {trainA.shape}')
print(f'trainB: {trainB.shape}')

trainA: (20437223, 26)
trainB: (19281955, 25)


In [2]:
# Anti-leakage cutoff: end of train period (UTC 2017-05-12 00:00).
# Aggregate features only use behavior data BEFORE this point,
# so val/test impressions cannot peek at concurrent user behaviors.
trainEnd = int(datetime(2017, 5, 12, 0, 0, 0, tzinfo=timezone.utc).timestamp())
validStart = int(datetime(2017, 4, 1).timestamp())

# One-pass group-by: compute 6 raw aggregates per user from behavior_log.
behaviorAgg = (
    pl.scan_parquet(DATA_DIR / 'behavior_log.parquet')
    .filter(
        (pl.col('time_stamp') >= validStart) &
        (pl.col('time_stamp') < trainEnd)
    )
    .group_by('user')
    .agg(
        user_total_events=pl.len(),
        user_total_pv=(pl.col('btag') == 'pv').sum(),
        user_total_buy=(pl.col('btag') == 'buy').sum(),
        user_total_cart=(pl.col('btag') == 'cart').sum(),
        user_total_fav=(pl.col('btag') == 'fav').sum(),
        user_n_unique_cates=pl.col('cate').n_unique(),
    )
    .collect()
)

# Global buy rate over the train-period behavior log; used as the prior
# for Bayesian smoothing of per-user buy rate.
totalBuy = behaviorAgg['user_total_buy'].sum()
totalEvents = behaviorAgg['user_total_events'].sum()
globalBuyRate = totalBuy / totalEvents
print(f'Global buy rate: {globalBuyRate:.4f}')

# Bayesian smoothing: pull low-activity users toward the global rate.
# alpha controls smoothing strength: smaller alpha = more user-specific,
# larger alpha = more global. alpha=10 is a typical CTR-modeling default.
ALPHA = 10
behaviorAgg = behaviorAgg.with_columns(
    user_buy_rate=(
        (pl.col('user_total_buy') + ALPHA * globalBuyRate) /
        (pl.col('user_total_events') + ALPHA)
    )
)

print(f'\nbehaviorAgg shape: {behaviorAgg.shape}')
print(behaviorAgg.head())

Global buy rate: 0.0127

behaviorAgg shape: (1132067, 8)
shape: (5, 8)
┌────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────┐
│ user   ┆ user_total ┆ user_total ┆ user_total ┆ user_total ┆ user_total ┆ user_n_uni ┆ user_buy_ │
│ ---    ┆ _events    ┆ _pv        ┆ _buy       ┆ _cart      ┆ _fav       ┆ que_cates  ┆ rate      │
│ i64    ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│        ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆ f64       │
╞════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════════╡
│ 952047 ┆ 137        ┆ 134        ┆ 0          ┆ 3          ┆ 0          ┆ 20         ┆ 0.000861  │
│ 469006 ┆ 201        ┆ 173        ┆ 23         ┆ 5          ┆ 0          ┆ 41         ┆ 0.109604  │
│ 881515 ┆ 335        ┆ 328        ┆ 6          ┆ 0          ┆ 1          ┆ 43         ┆ 0.017758  │
│ 566280 ┆ 16       

In [3]:
def joinBehaviorAgg(df, behaviorAgg, globalBuyRate):
    """Left-join per-user behavior aggregates onto an impression dataframe.

    Users absent from behavior_log (cold-start) get 0 for count features
    and the global prior for the rate feature.
    """
    return (
        df.join(behaviorAgg, on='user', how='left')
        .with_columns([
            pl.col('user_total_events').fill_null(0),
            pl.col('user_total_pv').fill_null(0),
            pl.col('user_total_buy').fill_null(0),
            pl.col('user_total_cart').fill_null(0),
            pl.col('user_total_fav').fill_null(0),
            pl.col('user_n_unique_cates').fill_null(0),
            pl.col('user_buy_rate').fill_null(globalBuyRate),
        ])
    )

trainA = joinBehaviorAgg(trainA, behaviorAgg, globalBuyRate)
valA   = joinBehaviorAgg(valA, behaviorAgg, globalBuyRate)
testA  = joinBehaviorAgg(testA, behaviorAgg, globalBuyRate)
trainB = joinBehaviorAgg(trainB, behaviorAgg, globalBuyRate)
valB   = joinBehaviorAgg(valB, behaviorAgg, globalBuyRate)
testB  = joinBehaviorAgg(testB, behaviorAgg, globalBuyRate)

print('Joined shapes:')
for name, df in [('trainA', trainA), ('valA', valA), ('testA', testA),
                  ('trainB', trainB), ('valB', valB), ('testB', testB)]:
    print(f'  {name}: {df.shape}')

Joined shapes:
  trainA: (20437223, 33)
  valA: (3271268, 33)
  testA: (2848377, 33)
  trainB: (19281955, 32)
  valB: (3078536, 32)
  testB: (2667885, 32)


In [4]:
# Sanity check: look at the new column distributions on trainA.
print('user_total_pv distribution (trainA):')
print(trainA.select('user_total_pv').describe())

# Overwrite the previously saved files in data/processed/wide/.
datasets = {'trainA': trainA, 'valA': valA, 'testA': testA,
            'trainB': trainB, 'valB': valB, 'testB': testB}

for name, df in datasets.items():
    path = WIDE / f'{name}.parquet'
    df.write_parquet(path, compression='snappy')
    sizeMb = path.stat().st_size / (1024**2)
    print(f'{name}: {df.shape} -> {sizeMb:.1f} MB')

user_total_pv distribution (trainA):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ user_total_pv │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 2.0437223e7   │
│ null_count ┆ 0.0           │
│ mean       ┆ 545.888547    │
│ std        ┆ 692.370509    │
│ min        ┆ 0.0           │
│ 25%        ┆ 135.0         │
│ 50%        ┆ 324.0         │
│ 75%        ┆ 691.0         │
│ max        ┆ 43133.0       │
└────────────┴───────────────┘
trainA: (20437223, 33) -> 730.8 MB
valA: (3271268, 33) -> 126.0 MB
testA: (2848377, 33) -> 107.7 MB
trainB: (19281955, 32) -> 687.4 MB
valB: (3078536, 32) -> 118.5 MB
testB: (2667885, 32) -> 100.9 MB
